In [1]:
import pymmcore_plus
from pymmcore_plus import CMMCorePlus
from useq import MDAEvent
from useq import MDASequence
from pathlib import Path
import json
import useq
import numpy as np
import tifffile as tiff
import datetime
import stages_movement
import CustomAcquisitionEngine
from stages_movement import controller
from VoiceCoil_nidaqmx import DAQ
import time
mmc = None
DAQ_VC = None

In [2]:
class MDA:
 
    def __init__(
        self,
        config_path: str = r"\Users\Hannah\Desktop\configuration\PVCAM_only.cfg"
    ):
        
        # Set the first instance of this class as the global singleton
        global mmc
        if mmc is not None:
            mmc.unloadAllDevices()
        if mmc is None:
            mmc = CMMCorePlus.instance()
        mmc.enableDebugLog(True)
        # Load the correct configuration file. NB! If a new configuration file is created, change the path to the new configuration file
        mmc.loadSystemConfiguration(config_path)
        mmc.setAutoShutter(False)
        #settings needed to set up the DAQ
        global DAQ_VC
        if DAQ_VC is None:
            DAQ_VC = DAQ()
        #Setup the default Camera settings for 1 fps rolling shutter.
        self._exposure  = 2.44
        self._scan_width = 8
        self._scan_direction = 'Up'
        self._trigger  = 'Edge Trigger'
        self._Port = 'Dynamic Range'
        self._save = False
        self._setup = False
        self._controller = None
        self._sequence = None
        self._cali_path = DAQ_VC.cali_path
        try:
            self._controller = controller
            self._controller.connect()
        except Exception as e:
            print(f'The controller could not connect to the stage: {e}')
        self._boundary = np.array([[-40, 20], [0.5, 18],[-0.5, 5]])
        self._home = np.array([-20, 9, 0])
        
    @property
    def save(self) -> bool:
        return getattr(self,"_save",None)
    @save.setter
    def save(self, value: bool):
        self._save = value

    @property
    def exposure(self) -> float:
        return getattr(self,"_exposure",None)
    @exposure.setter
    def exposure(self, value: float):
        self._exposure = value
    
    @property
    def scan_width(self) -> int:
        return getattr(self,"_scan_width",None)
    @scan_width.setter
    def scan_width(self, value: int):
        self._scan_width = value
        
    @property
    def cali_path(self) -> str:
        return getattr(self,"_cali_path",None)
    @cali_path.setter
    def cali_path(self, value: str):
        DAQ_VC.cali_path = value
        self._cali_path = DAQ_VC.cali_path            
    
    def connect_stages(self):
        try:
            self._controller.connect()
        except Exception as e:
            print(f'The controller could not connect to the stage: {e}')

        
    def setup_save(
        self,
        silence: bool = False,
        filename: str = 'Zstack',
        foldername: str = 'Default'
    ):
        self._silence = silence
        self._filename = filename
        self._foldername = foldername
        self._tif = None
        self._stack = None
        
        try:
            mmc.mda.events.frameReady.disconnect(on_frame)
        except Exception:
            pass
        @mmc.mda.events.frameReady.connect
        def on_frame(image: np.ndarray, event: useq.MDAEvent, meta: pymmcore_plus.metadata.FrameMetaV1):
            t_start = time.perf_counter()
            if self._silence == False:
                print(
                    f"received frame:{event.metadata['idx']}" 
                )
            if self._save:
                t_write_start = time.perf_counter()
                self._tif.write(image, photometric='minisblack')
                t_write_end = time.perf_counter()
                print(f"Saving took{(t_write_end-t_write_start)*1000}, total time is: {(t_write_end-t_start)*1000}")
                if int(event.metadata['idx']) == (int(event.metadata['stack_height'])*self._stack) - 1:
                    self._tif.close()
                    self._stack += 1
                    self._tif = tiff.TiffWriter(f"{self._tif_path}{self._stack:05d}.tif")
                    
                
        try:
            mmc.mda.events.sequenceStarted.disconnect(sequenceStart)
        except Exception:
            pass        
        @mmc.mda.events.sequenceStarted.connect
        def sequenceStart(image: np.ndarray, event: useq.MDAEvent):
            
            
            if self._save:
                today =datetime.datetime.now()   # Get date
                self._datestring = today.strftime("%Y-%m-%d")  # Date to the desired string format
                Path(self._datestring).mkdir(parents=True, exist_ok=True)   # Create folder
                
                if self._foldername == 'Default':
                    time_m_s = today.strftime("%H_%M")
                    Path(f"{self._datestring}\\{time_m_s}").mkdir(parents=True, exist_ok=True)
                    self._tif_path = f"{Path(self._datestring)}\\{time_m_s}\\{self._filename}"
                else:
                    Path(f"{self._datestring}\\{self._foldername}").mkdir(parents=True, exist_ok=True)
                    self._tif_path = f"{Path(self._datestring)}\\{self._foldername}\\{self._filename}"
                
                self._stack = 1
                #json.dump(event,open(f"{self._tif_path}_metadata.json",'w'),indent = 4)
                self._tif = tiff.TiffWriter(f"{self._tif_path}{self._stack:05d}.tif")
            if self._silence == False:
                if self._save:
                    print(
                        f"Sequence started. Data will be saved in: {self._tif_path}" 
                    )
                else:
                    print(f"Sequence started. Data is not saved.")
        try:
            mmc.mda.events.sequenceFinished.disconnect(sequenceDone)
        except Exception:
            pass                
        @mmc.mda.events.sequenceFinished.connect
        def sequenceDone(p0: useq._mda_sequence.MDASequence, /):
            DAQ_VC.stop()
            if self._save == True:
                try:
                    self._tif.close()
                except Exception:
                      pass
            if self._silence == False:
                if self._save:
                    print(
                        f"Sequence finished. Data is saved in: {self._tif_path}" 
                    )
                else:
                    print(f"Sequence finished.")
                
        try:
            mmc.mda.events.sequenceCanceled.disconnect(sequenceCancel)
        except Exception:
            pass                
        @mmc.mda.events.sequenceCanceled.connect
        def sequenceCancel(p0: useq._mda_sequence.MDASequence, /):
            if self._save == True:
                try:
                    self._tif.close()
                except Exception:
                      pass
            if self._silence == False:
                if self._save:
                    print(
                        f"Sequence cancelled. Data is saved in: {self._tif_path}." 
                    )
                else:
                    print(f"Sequence cancelled.")
    
    def setup_sequence(
        self,
        z_depth: float,
        x_tiles: int = 0,
        y_tiles: int = 0,
        z_stepsize: float = 0.2,
        overlap: float = 5
    ):
        stack_height = round(z_depth / (z_stepsize/1000))
        
        if self._trigger == 'Edge Trigger':
            DAQ_VC.program_waveforms()
        if x_tiles == 0 and y_tiles == 0:
            mmc.setProperty('Camera-1','Exposure',self._exposure),
            mmc.setProperty('Camera-1','TriggerMode',self._trigger)
            mmc.setProperty('Camera-1','ScanDirection',self._scan_direction)
            mmc.setProperty('Camera-1','ScanMode','Scan Width'),
            mmc.setProperty('Camera-1','ScanWidth',self._scan_width)
            mmc.setProperty('Camera-1','Port',self._Port)
            mmc.setProperty('Camera-1','ShutterMode','Never')

            try:
                #mmc.mda.set_engine(engine = CustomAcquisitionEngine.Z_Stack(mmc,z_stepsize/1000,self._controller,self._boundary))
                #mmc.mda.set_engine(engine = CustomAcquisitionEngine.No_Stage(mmc))
                mda_sequence = []
                for i in range(stack_height):
                    mda_sequence += [
                        MDAEvent(
                             metadata = {
                                 'idx': str(i),
                                 'stack_height': str(stack_height)
                             },keep_shutter_open = True
                        )
                    ]
                self._sequence = mda_sequence
                self._setup = True
            except Exception as e:
                print(f"Sequence generation failed: {e}")
    
    def run_sequence(self):
        if self._setup == False:
            return('A sequence has not been defined yet')
        else:
            mmc.mda.engine.use_hardware_sequencing = True
            mmc.run_mda(iter(self._sequence))
            if self._trigger == 'Edge Trigger':
                DAQ_VC.start()
            
    def stop_sequence(self):
        if mmc.mda.is_running():
            mmc.mda.cancel()
        else:
            print('No sequence is running')
    def set_home(
        self
    ):
        ax0 = self._controller.axes[0]
        ax1 = self._controller.axes[1]
        ax2 = self._controller.axes[2]
        
        ax0_bounds = [-40, 2]
        ax1_bounds = np.zeros(2)
        ax2_bounds = [-0.5, 5]
        self._boundary = None
        self._home = None
        
        if ax1.enabled==True:
            ax1.disable()
        if ax0.enabled==True:
            ax0.disable()
        if ax2.enabled==False:
            ax2.enable()
        
        for i in range(2):
            input('Move the stage to one edge')
            ax1_bounds[i] = ax1.rpos
        
        self._controller.enable_all()
        
        ax1_bounds_sorted = np.sort(ax1_bounds)
        ax0_home = ax0.rpos
        ax1_home = (ax1_bounds[1] + ax1_bounds[0])/2
        ax2_home = 0
        self._home = np.array([ax0_home, ax1_home, ax2_home])
        self._boundary = np.array([ax0_bounds, ax1_bounds_sorted, ax2_bounds])
        
    def move_home(self):
        stages_movement.move_to(self._home, self._controller,self._boundary)
        
    def disable_x(self):
        ax0 = self._controller.axes[0]
        ax0.disable()
        
    def enable_x(self):
        ax0 = self._controller.axes[0]
        ax0.enable()
        
    def disable_y(self):
        ax1 = self._controller.axes[1]
        ax1.disable()
        
    def enable_y(self):
        ax1 = self._controller.axes[1]
        ax1.enable()
        
    def disable_z(self):
        ax2 = self._controller.axes[2]
        ax2.disable()
        
    def enable_z(self):
        ax2 = self._controller.axes[2]
        ax2.enable()

            
    def close(self):
        mmc.unloadAllDevices()
        try:
            DAQ_VC.close()
        except Exceptions as e:
            print(f"Could not close DAQ: {e}")
        try:
            self._controller.disconnect()
        except Exception as e:
            print(f"Could not disconnect stage: {e}")

In [3]:
test = MDA()

OSError: Line 18: Property,Core,Initialize,1
Error in device "Camera-1": [PVCAM] ERR: pl_cam_open failed, pvErr:195, pvMsg:'Driver device failed to open (PL_ERR_DDI_DEVICE_OPEN_FAILED)' (20195)



In [ ]:
test.setup_save()

In [ ]:
test.save = True

In [ ]:
test.setup_sequence(z_depth = 0.01)

In [ ]:
test.run_sequence()

In [ ]:
test.stop_sequence()


In [ ]:
test.move_home()

In [ ]:
test.disable_x()
test.disable_y()
test.disable_z()

In [4]:
test.close()

NameError: name 'test' is not defined

In [ ]:
import acspy as acs
t_start = time.perf_counter()
acs.acsc.toPoint(controller.hc, 0, 2, 0.2/1000)

while True:
    if acs.acsc.getMotorState(controller.hc,2)['in position']:
        t_end = time.perf_counter()
        print(f"Stages took {(t_end-t_start)*1000} to move")
        break
    else: 
        if (time.perf_counter()-t_start) > 10:
            break

In [ ]:
acs.acsc.halt(controller.hc,2)

In [ ]:
acs.acsc.getVelocity(controller.hc,2)

In [ ]:
for prop in mmc.getDevicePropertyNames("Camera-1"):  # adjust device name as needed
    print(mmc.getAllowedPropertyValues("Camera-1", prop))
    print(prop, ":", mmc.getProperty("Camera-1", prop))

In [ ]:
mmc.setProperty('Camera-1','Timing-ReadoutTimeNs',12000)
print(mmc.getProperty('Camera-1','Timing-ReadoutTimeNs'))

In [ ]:
mmc.startSequenceAcquisition(
    10,  # number of frames
    0,             # intervalMS - 0 means as fast as possible
    True           # stopOnOverflow
)
DAQ_VC.start()

In [ ]:
DAQ_VC.stop()

In [ ]:
mmc.stopSequenceAcquisition()